# Neural Turing Machines: Learning to Use External Memory

Neural networks traditionally store information in their weights during training. But what if a network could learn to **read from and write to external memory** like a computer program? This is the core idea behind **Neural Turing Machines (NTMs)**.

## What You'll Learn

- Why external memory matters for neural networks
- How content-based addressing works (like attention)
- How location-based addressing enables sequential access
- Building read and write mechanisms from scratch
- Training an NTM on the copy task
- Visualizing learned memory access patterns

## The Key Insight

NTMs combine three powerful ideas:
1. **Differentiable memory**: Memory that can be trained with backpropagation
2. **Attention-based addressing**: Focus on relevant memory locations
3. **Location-based navigation**: Move through memory sequentially like a Turing machine

This architecture can learn algorithms that require explicit memory manipulation, like copying, sorting, and associative recall.

## 1. Setup

### Configuration

All hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducibility
    
    # Data
    'batch_size': 32,  # Number of sequences per batch
    'num_workers': 0,  # DataLoader workers (0 for main thread)
    'seq_length': 8,  # Length of sequences to copy
    'vector_size': 6,  # Size of each vector in the sequence
    'num_train_samples': 1000,  # Training samples
    'num_val_samples': 200,  # Validation samples
    
    # Training
    'learning_rate': 1e-3,  # Adam optimizer learning rate
    'max_epochs': 15,  # Maximum training epochs
    'early_stop_patience': 5,  # Patience for early stopping
    'gradient_clip_val': 10.0,  # Gradient clipping threshold
    
    # NTM Architecture
    'controller_hidden_size': 64,  # LSTM controller hidden units
    'memory_size': 64,  # Number of memory locations (N)
    'memory_vector_size': 16,  # Size of each memory vector (M)
    'num_heads': 1,  # Number of read/write heads
    'shift_range': 1,  # Allowed shift range for location addressing
}

### Random Seed & Device

Set random seed for reproducibility and select the best available device.

In [ ]:
from aiml_notebooks import set_seed, get_device

%load_ext autoreload
%autoreload 2

set_seed(CONFIG['seed'])
device = get_device()
print(f"Using device: {device}")

## 2. The Problem: Why External Memory?

Traditional RNNs store information in their hidden state. This has two limitations:

1. **Fixed capacity**: The hidden state size is fixed, limiting how much information can be stored
2. **Implicit storage**: The network doesn't explicitly choose where to store information

**Neural Turing Machines** solve this by adding an external memory matrix that the network learns to read from and write to explicitly.

### Visualize the concept

Let's visualize what an NTM memory matrix looks like.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Create a simple memory matrix
N, M = 8, 5  # 8 locations, 5 values per location
memory = torch.randn(N, M)

# Create addressing weights (where the head is looking)
weights = torch.softmax(torch.randn(N), dim=0)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Memory matrix
im = ax1.imshow(memory.numpy(), cmap='RdBu', aspect='auto')
ax1.set_xlabel('Memory Vector Dimensions', fontsize=11)
ax1.set_ylabel('Memory Locations', fontsize=11)
ax1.set_title('Memory Matrix (N×M)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax1)

# Addressing weights
ax2.barh(range(N), weights.numpy(), color='#4ECDC4')
ax2.set_xlabel('Attention Weight', fontsize=11)
ax2.set_ylabel('Memory Location', fontsize=11)
ax2.set_title('Read Head Weights', fontsize=13, fontweight='bold')
ax2.set_ylim(-0.5, N-0.5)
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Memory shape: {memory.shape}")
print(f"Head weights shape: {weights.shape}")
print(f"Head weights sum to: {weights.sum():.4f} (should be 1.0)")

**Key insight**: The head uses **soft attention** (weights sum to 1) to read from memory. This makes the operation differentiable and trainable.

## 3. Building Block 1: Content-Based Addressing

Content-based addressing lets the network find memory locations by their content, similar to attention mechanisms. Given a **key vector**, we compute similarity to all memory locations.

### The cosine similarity formula

We measure similarity using cosine similarity with a sharpening parameter β:

$$w^c_i = \frac{\exp(\beta \cdot \text{cosine}(k, M_i))}{\sum_j \exp(\beta \cdot \text{cosine}(k, M_j))}$$

Where:
- $k$ is the key vector (what we're looking for)
- $M_i$ is memory location $i$
- $\beta$ is the key strength (higher = sharper focus)

### Implement content addressing

Let's implement this mechanism from scratch.

In [ ]:
import torch.nn.functional as F

def content_addressing(memory, key, beta):
    """
    Content-based addressing using cosine similarity.
    
    Args:
        memory: (batch, N, M) - Memory matrix
        key: (batch, M) - Key vector to search for
        beta: (batch, 1) - Key strength (sharpening parameter)
    
    Returns:
        weights: (batch, N) - Attention weights over memory locations
    """
    # Normalize key and memory for cosine similarity
    key = key.unsqueeze(1)  # (batch, 1, M)
    
    # Compute cosine similarity: (batch, N)
    similarity = F.cosine_similarity(key, memory, dim=2)
    
    # Apply key strength and softmax
    weights = F.softmax(beta * similarity, dim=1)
    
    return weights

# Test it
batch_size = 2
N, M = 8, 5

test_memory = torch.randn(batch_size, N, M)
test_key = torch.randn(batch_size, M)
test_beta = torch.tensor([[3.0], [10.0]])  # Different strengths per batch

weights = content_addressing(test_memory, test_key, test_beta)

print(f"Input shapes:")
print(f"  Memory: {test_memory.shape}")
print(f"  Key: {test_key.shape}")
print(f"  Beta: {test_beta.shape}")
print(f"\nOutput weights shape: {weights.shape}")
print(f"Weights sum: {weights[0].sum():.4f}, {weights[1].sum():.4f}")
print(f"\nBatch 0 (β=3): {weights[0].numpy()}")
print(f"Batch 1 (β=10): {weights[1].numpy()}")
print(f"\nNotice: Higher β creates sharper focus (less uniform distribution)")

### Visualize the effect of β

The key strength β controls how focused the attention is.

In [ ]:
# Create memory and key
memory = torch.randn(1, 10, 8)
key = torch.randn(1, 8)

# Try different β values
betas = [0.5, 1.0, 3.0, 10.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, beta_val in zip(axes, betas):
    beta = torch.tensor([[beta_val]])
    weights = content_addressing(memory, key, beta)
    
    ax.bar(range(10), weights[0].numpy(), color='#4ECDC4')
    ax.set_xlabel('Memory Location', fontsize=10)
    ax.set_ylabel('Weight', fontsize=10)
    ax.set_title(f'β = {beta_val}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: Higher β creates sharper, more focused attention.")
print("Lower β distributes attention more uniformly across similar locations.")

## 4. Building Block 2: Location-Based Addressing

Content addressing finds locations by similarity. But sometimes we want to **move sequentially** through memory (like reading a list). Location-based addressing enables this through:

1. **Interpolation**: Blend previous and current weights
2. **Convolutional shift**: Move the attention focus
3. **Sharpening**: Make the distribution more peaked

### Step 1: Interpolation

Blend between content-based weights and previous weights:

$$w^g = g \cdot w^c + (1-g) \cdot w_{prev}$$

Where $g \in [0,1]$ is the interpolation gate.

In [ ]:
def interpolate_weights(w_content, w_prev, g):
    """
    Interpolate between content-based and previous weights.
    
    Args:
        w_content: (batch, N) - Content-based weights
        w_prev: (batch, N) - Previous weights
        g: (batch, 1) - Interpolation gate in [0,1]
    
    Returns:
        w_gated: (batch, N) - Interpolated weights
    """
    return g * w_content + (1 - g) * w_prev

# Test it
w_content = torch.tensor([[0.1, 0.9, 0.0, 0.0, 0.0]])
w_prev = torch.tensor([[0.0, 0.0, 0.0, 0.8, 0.2]])

fig, axes = plt.subplots(1, 3, figsize=(15, 3))

for ax, g_val in zip(axes, [0.0, 0.5, 1.0]):
    g = torch.tensor([[g_val]])
    w_gated = interpolate_weights(w_content, w_prev, g)
    
    x = np.arange(5)
    width = 0.35
    ax.bar(x - width/2, w_content[0].numpy(), width, label='Content', color='#4ECDC4')
    ax.bar(x + width/2, w_prev[0].numpy(), width, label='Previous', color='#FF6B6B')
    ax.plot(x, w_gated[0].numpy(), 'o-', linewidth=3, markersize=8, 
            label='Interpolated', color='#95E1D3')
    ax.set_xlabel('Memory Location', fontsize=10)
    ax.set_ylabel('Weight', fontsize=10)
    ax.set_title(f'g = {g_val}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("g=0: Use only previous weights (ignore content)")
print("g=1: Use only content weights (ignore previous)")
print("g=0.5: Blend both equally")

### Step 2: Convolutional Shift

Shift the attention focus left or right using a learned shift distribution:

$$w^s_i = \sum_j w^g_j s(i-j)$$

Where $s$ is the shift weighting (e.g., [-1, 0, +1] for shifts).

In [ ]:
def circular_convolution(w, s):
    """
    Apply circular convolution for shifting weights.
    
    Args:
        w: (batch, N) - Weights to shift
        s: (batch, shift_range*2+1) - Shift distribution
    
    Returns:
        w_shifted: (batch, N) - Shifted weights
    """
    batch_size, N = w.shape
    shift_size = s.shape[1]
    shift_range = shift_size // 2
    
    # Initialize output
    w_shifted = torch.zeros_like(w)
    
    # Apply each shift with its weight
    for i in range(shift_size):
        shift_amount = i - shift_range
        w_shifted += s[:, i:i+1] * torch.roll(w, shifts=shift_amount, dims=1)
    
    return w_shifted

# Test it
w = torch.tensor([[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]])  # Focused at position 2

# Different shift distributions
shift_left = torch.tensor([[0.9, 0.1, 0.0]])     # Shift left
shift_none = torch.tensor([[0.0, 1.0, 0.0]])     # No shift
shift_right = torch.tensor([[0.0, 0.1, 0.9]])    # Shift right

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
shifts = [shift_left, shift_none, shift_right]
titles = ['Shift Left', 'No Shift', 'Shift Right']

for ax, s, title in zip(axes, shifts, titles):
    w_shifted = circular_convolution(w, s)
    
    ax.bar(range(8), w[0].numpy(), alpha=0.3, label='Original', color='gray')
    ax.bar(range(8), w_shifted[0].numpy(), label='Shifted', color='#4ECDC4')
    ax.set_xlabel('Memory Location', fontsize=10)
    ax.set_ylabel('Weight', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1.2)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Convolutional shift enables sequential memory access.")
print("The network learns where to move the attention focus.")

### Step 3: Sharpening

Make the distribution more focused:

$$w_i = \frac{w_i^{\gamma}}{\sum_j w_j^{\gamma}}$$

Where $\gamma \geq 1$ is the sharpening parameter.

In [ ]:
def sharpen_weights(w, gamma):
    """
    Sharpen weight distribution.
    
    Args:
        w: (batch, N) - Weights to sharpen
        gamma: (batch, 1) - Sharpening parameter >= 1
    
    Returns:
        w_sharpened: (batch, N) - Sharpened weights
    """
    w_sharp = w ** gamma
    return w_sharp / (w_sharp.sum(dim=1, keepdim=True) + 1e-8)

# Test it
w = torch.tensor([[0.15, 0.35, 0.25, 0.15, 0.10]])

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
gammas = [1.0, 2.0, 5.0, 10.0]

for ax, gamma_val in zip(axes, gammas):
    gamma = torch.tensor([[gamma_val]])
    w_sharp = sharpen_weights(w, gamma)
    
    ax.bar(range(5), w_sharp[0].numpy(), color='#4ECDC4')
    ax.axhline(y=0.2, color='red', linestyle='--', alpha=0.5, label='Uniform')
    ax.set_xlabel('Memory Location', fontsize=10)
    ax.set_ylabel('Weight', fontsize=10)
    ax.set_title(f'γ = {gamma_val}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Sharpening makes the distribution more peaked.")
print("γ=1: No change | γ>1: Increasingly focused")

## 5. Building Block 3: Reading from Memory

Reading is simple: compute a weighted sum of memory locations.

$$r = \sum_i w_i M_i$$

Where $w$ are the addressing weights and $M$ is memory.

### Implement the read operation

This is just a weighted sum over the memory dimension.

In [ ]:
def read_memory(memory, weights):
    """
    Read from memory using attention weights.
    
    Args:
        memory: (batch, N, M) - Memory matrix
        weights: (batch, N) - Read weights
    
    Returns:
        read_vector: (batch, M) - Read content
    """
    # Weighted sum: (batch, N, 1) * (batch, N, M) -> (batch, N, M) -> (batch, M)
    return torch.sum(weights.unsqueeze(2) * memory, dim=1)

# Test it
memory = torch.randn(2, 8, 5)  # 2 batches, 8 locations, 5 dims

# Sharp focus on location 3
weights = torch.zeros(2, 8)
weights[:, 3] = 1.0

read_vec = read_memory(memory, weights)

print(f"Memory shape: {memory.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Read vector shape: {read_vec.shape}")
print(f"\nMemory at location 3:\n{memory[0, 3]}")
print(f"\nRead vector (should match):\n{read_vec[0]}")
print(f"\nDifference: {(memory[0, 3] - read_vec[0]).abs().max():.6f}")

## 6. Building Block 4: Writing to Memory

Writing has two operations:

1. **Erase**: Remove old content using an erase vector $e \in [0,1]^M$
2. **Add**: Write new content using an add vector $a \in \mathbb{R}^M$

$$M_i^{new} = M_i^{old} \odot (1 - w_i e) + w_i a$$

### Implement the write operation

Erase, then add.

In [ ]:
def write_memory(memory, weights, erase_vector, add_vector):
    """
    Write to memory: erase then add.
    
    Args:
        memory: (batch, N, M) - Memory matrix
        weights: (batch, N) - Write weights
        erase_vector: (batch, M) - What to erase (in [0,1])
        add_vector: (batch, M) - What to add
    
    Returns:
        memory_new: (batch, N, M) - Updated memory
    """
    # Expand dimensions for broadcasting
    w = weights.unsqueeze(2)  # (batch, N, 1)
    e = erase_vector.unsqueeze(1)  # (batch, 1, M)
    a = add_vector.unsqueeze(1)  # (batch, 1, M)
    
    # Erase: M * (1 - w*e)
    memory_erased = memory * (1 - w * e)
    
    # Add: M_erased + w*a
    memory_new = memory_erased + w * a
    
    return memory_new

# Test it
memory = torch.ones(1, 5, 3)  # All ones initially
weights = torch.zeros(1, 5)
weights[0, 2] = 1.0  # Write to location 2

erase_vector = torch.ones(1, 3)  # Erase everything
add_vector = torch.tensor([[5.0, 5.0, 5.0]])  # Write 5s

memory_new = write_memory(memory, weights, erase_vector, add_vector)

print("Before write:")
print(memory[0])
print("\nAfter write (location 2 should be [5, 5, 5]):")
print(memory_new[0])

# Test partial erase
erase_partial = torch.tensor([[1.0, 0.0, 0.5]])  # Erase dim 0 fully, dim 2 half
memory_partial = write_memory(memory, weights, erase_partial, add_vector)

print("\nWith partial erase [1.0, 0.0, 0.5]:")
print(memory_partial[0])
print("Dim 0: fully erased+added = 5.0")
print("Dim 1: not erased, still 1.0 (no erase)")
print("Dim 2: half erased = 0.5 + added 5.0 = 5.5")

### Visualize write operation

Let's see how erase and add affect memory.

In [ ]:
# Create memory with pattern
memory = torch.randn(1, 8, 6)

# Write to locations 2 and 5
weights = torch.zeros(1, 8)
weights[0, [2, 5]] = 0.8
weights = weights / weights.sum()

erase_vector = torch.ones(1, 6)
add_vector = torch.full((1, 6), 3.0)

memory_new = write_memory(memory, weights, erase_vector, add_vector)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original memory
im0 = axes[0].imshow(memory[0].numpy(), cmap='RdBu', aspect='auto', vmin=-3, vmax=3)
axes[0].set_xlabel('Memory Dimensions', fontsize=10)
axes[0].set_ylabel('Memory Locations', fontsize=10)
axes[0].set_title('Original Memory', fontsize=12, fontweight='bold')
plt.colorbar(im0, ax=axes[0])

# Write weights
axes[1].barh(range(8), weights[0].numpy(), color='#4ECDC4')
axes[1].set_xlabel('Write Weight', fontsize=10)
axes[1].set_ylabel('Memory Location', fontsize=10)
axes[1].set_title('Write Head Weights', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

# New memory
im2 = axes[2].imshow(memory_new[0].numpy(), cmap='RdBu', aspect='auto', vmin=-3, vmax=3)
axes[2].set_xlabel('Memory Dimensions', fontsize=10)
axes[2].set_ylabel('Memory Locations', fontsize=10)
axes[2].set_title('After Write', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

print("Locations 2 and 5 have been partially overwritten with value 3.0")

## 7. Putting It Together: The NTM Head

Now we combine all addressing mechanisms into a single NTM head that can read or write. The head takes controller outputs and produces addressing weights.

### Implement the complete NTM head

This module combines content and location addressing.

In [ ]:
import torch.nn as nn

class NTMHead(nn.Module):
    """NTM Read/Write Head with content and location addressing."""
    
    def __init__(self, memory_vector_size, controller_size, shift_range=1):
        super().__init__()
        self.memory_vector_size = memory_vector_size
        self.shift_range = shift_range
        self.shift_size = 2 * shift_range + 1
        
        # Parameters for addressing
        num_params = (
            memory_vector_size +  # key
            1 +  # beta (key strength)
            1 +  # g (interpolation gate)
            self.shift_size +  # shift weights
            1  # gamma (sharpening)
        )
        
        self.fc_params = nn.Linear(controller_size, num_params)
        
    def forward(self, controller_output, memory, prev_weights):
        """Compute addressing weights.
        
        Args:
            controller_output: (batch, controller_size)
            memory: (batch, N, M)
            prev_weights: (batch, N)
        
        Returns:
            weights: (batch, N) - Addressing weights
        """
        # Generate addressing parameters
        params = self.fc_params(controller_output)
        
        # Split parameters
        M = self.memory_vector_size
        key = params[:, :M]
        beta = F.softplus(params[:, M:M+1])  # Key strength > 0
        g = torch.sigmoid(params[:, M+1:M+2])  # Gate in [0,1]
        shift = F.softmax(params[:, M+2:M+2+self.shift_size], dim=1)
        gamma = 1 + F.softplus(params[:, -1:])  # Sharpening >= 1
        
        # 1. Content addressing
        w_content = content_addressing(memory, key, beta)
        
        # 2. Interpolation
        w_gated = interpolate_weights(w_content, prev_weights, g)
        
        # 3. Convolutional shift
        w_shifted = circular_convolution(w_gated, shift)
        
        # 4. Sharpening
        weights = sharpen_weights(w_shifted, gamma)
        
        return weights

# Test it
head = NTMHead(
    memory_vector_size=CONFIG['memory_vector_size'],
    controller_size=CONFIG['controller_hidden_size'],
    shift_range=CONFIG['shift_range']
)

batch_size = 4
controller_output = torch.randn(batch_size, CONFIG['controller_hidden_size'])
memory = torch.randn(batch_size, CONFIG['memory_size'], CONFIG['memory_vector_size'])
prev_weights = torch.softmax(torch.randn(batch_size, CONFIG['memory_size']), dim=1)

weights = head(controller_output, memory, prev_weights)

print(f"Controller output: {controller_output.shape}")
print(f"Memory: {memory.shape}")
print(f"Previous weights: {prev_weights.shape}")
print(f"Output weights: {weights.shape}")
print(f"Weights sum: {weights[0].sum():.6f} (should be 1.0)")
print(f"\nHead has {sum(p.numel() for p in head.parameters())} parameters")

## 8. The Complete NTM Architecture

Now we build the full NTM with:
- **Controller**: LSTM that processes input and read vectors
- **Read heads**: Generate addressing weights and read from memory
- **Write head**: Generate weights, erase vector, and add vector
- **Output**: Combine controller state and read vectors

### Implement the full NTM

This is the complete Neural Turing Machine.

In [ ]:
import lightning as L

class NeuralTuringMachine(L.LightningModule):
    """Complete Neural Turing Machine."""
    
    def __init__(
        self,
        input_size,
        output_size,
        controller_hidden_size=CONFIG['controller_hidden_size'],
        memory_size=CONFIG['memory_size'],
        memory_vector_size=CONFIG['memory_vector_size'],
        num_heads=CONFIG['num_heads'],
        shift_range=CONFIG['shift_range'],
        learning_rate=CONFIG['learning_rate']
    ):
        super().__init__()
        self.save_hyperparameters()
        
        self.input_size = input_size
        self.output_size = output_size
        self.controller_hidden_size = controller_hidden_size
        self.memory_size = memory_size
        self.memory_vector_size = memory_vector_size
        self.num_heads = num_heads
        
        # Controller: LSTM that takes input + read vectors
        controller_input_size = input_size + num_heads * memory_vector_size
        self.controller = nn.LSTM(controller_input_size, controller_hidden_size, batch_first=True)
        
        # Read heads
        self.read_heads = nn.ModuleList([
            NTMHead(memory_vector_size, controller_hidden_size, shift_range)
            for _ in range(num_heads)
        ])
        
        # Write head
        self.write_head = NTMHead(memory_vector_size, controller_hidden_size, shift_range)
        
        # Write parameters (erase and add vectors)
        self.fc_erase = nn.Linear(controller_hidden_size, memory_vector_size)
        self.fc_add = nn.Linear(controller_hidden_size, memory_vector_size)
        
        # Output layer
        output_input_size = controller_hidden_size + num_heads * memory_vector_size
        self.fc_output = nn.Linear(output_input_size, output_size)
        
        self.criterion = nn.BCEWithLogitsLoss()
        
    def init_state(self, batch_size):
        """Initialize memory and weights."""
        device = next(self.parameters()).device
        
        # Initialize memory to small random values
        memory = torch.randn(
            batch_size, self.memory_size, self.memory_vector_size,
            device=device
        ) * 0.01
        
        # Initialize all heads to uniform attention
        read_weights = [
            torch.ones(batch_size, self.memory_size, device=device) / self.memory_size
            for _ in range(self.num_heads)
        ]
        write_weights = torch.ones(batch_size, self.memory_size, device=device) / self.memory_size
        
        return memory, read_weights, write_weights
    
    def forward(self, x):
        """Process sequence.
        
        Args:
            x: (batch, seq_len, input_size)
        
        Returns:
            outputs: (batch, seq_len, output_size)
        """
        batch_size, seq_len, _ = x.shape
        
        # Initialize
        memory, read_weights, write_weights = self.init_state(batch_size)
        controller_state = None
        
        outputs = []
        
        for t in range(seq_len):
            # Read from memory
            reads = [read_memory(memory, w) for w in read_weights]
            
            # Controller input: current input + read vectors
            controller_input = torch.cat([x[:, t], *reads], dim=1).unsqueeze(1)
            
            # Controller forward
            controller_output, controller_state = self.controller(controller_input, controller_state)
            controller_output = controller_output.squeeze(1)
            
            # Update read heads
            read_weights = [
                head(controller_output, memory, prev_w)
                for head, prev_w in zip(self.read_heads, read_weights)
            ]
            
            # Update write head
            write_weights = self.write_head(controller_output, memory, write_weights)
            
            # Write to memory
            erase = torch.sigmoid(self.fc_erase(controller_output))
            add = torch.tanh(self.fc_add(controller_output))
            memory = write_memory(memory, write_weights, erase, add)
            
            # Generate output
            reads = [read_memory(memory, w) for w in read_weights]
            output_input = torch.cat([controller_output, *reads], dim=1)
            output = self.fc_output(output_input)
            outputs.append(output)
        
        return torch.stack(outputs, dim=1)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        
        # Binary accuracy
        acc = ((torch.sigmoid(y_hat) > 0.5) == y).float().mean()
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        
        acc = ((torch.sigmoid(y_hat) > 0.5) == y).float().mean()
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

# Create model
model = NeuralTuringMachine(
    input_size=CONFIG['vector_size'] + 1,  # +1 for delimiter
    output_size=CONFIG['vector_size']
)

from aiml_notebooks import count_parameters, print_model_summary
print_model_summary(model)
print(f"\nTotal parameters: {count_parameters(model):,}")

## 9. The Copy Task

We'll train the NTM on the **copy task**: given a sequence of random binary vectors, store them in memory, then reproduce them exactly.

**Input format**: `[v1, v2, ..., vN, delimiter, 0, 0, ..., 0]`

**Target output**: `[0, 0, ..., 0, 0, v1, v2, ..., vN]`

The network must learn to:
1. Write vectors to memory during input phase
2. Detect the delimiter
3. Read vectors from memory in order during output phase

### Create the copy task dataset

Generate random binary sequences with a delimiter.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CopyTaskDataset(Dataset):
    """Copy task: memorize and reproduce a sequence."""
    
    def __init__(self, num_samples, seq_length, vector_size):
        self.num_samples = num_samples
        self.seq_length = seq_length
        self.vector_size = vector_size
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Generate random binary sequence
        sequence = torch.randint(0, 2, (self.seq_length, self.vector_size)).float()
        
        # Create input: [sequence, delimiter, zeros]
        delimiter = torch.zeros(1, self.vector_size + 1)
        delimiter[0, -1] = 1.0  # Delimiter in extra dimension
        
        zeros_input = torch.zeros(self.seq_length, self.vector_size + 1)
        sequence_input = torch.cat([sequence, torch.zeros(self.seq_length, 1)], dim=1)
        
        x = torch.cat([sequence_input, delimiter, zeros_input], dim=0)
        
        # Create target: [zeros, sequence]
        zeros_target = torch.zeros(self.seq_length + 1, self.vector_size)
        y = torch.cat([zeros_target, sequence], dim=0)
        
        return x, y

# Create datasets
train_dataset = CopyTaskDataset(
    CONFIG['num_train_samples'],
    CONFIG['seq_length'],
    CONFIG['vector_size']
)

val_dataset = CopyTaskDataset(
    CONFIG['num_val_samples'],
    CONFIG['seq_length'],
    CONFIG['vector_size']
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers']
)

# Show example
x, y = train_dataset[0]
print(f"Sequence length: {CONFIG['seq_length']}")
print(f"Vector size: {CONFIG['vector_size']}")
print(f"Input shape: {x.shape}  (seq_len + 1 + seq_len, vector_size + 1)")
print(f"Target shape: {y.shape}  (seq_len + 1 + seq_len, vector_size)")
print(f"\nTotal sequence length: {x.shape[0]} = {CONFIG['seq_length']} (input) + 1 (delimiter) + {CONFIG['seq_length']} (output)")

### Visualize a sample from the copy task

Let's see what the task looks like.

In [ ]:
x, y = train_dataset[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

# Input
im1 = ax1.imshow(x.T.numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax1.axvline(x=CONFIG['seq_length'] - 0.5, color='red', linestyle='--', linewidth=2, label='Delimiter')
ax1.set_ylabel('Input Dimensions\n(last=delimiter)', fontsize=11)
ax1.set_title('Input Sequence', fontsize=13, fontweight='bold')
ax1.set_xticks([])
ax1.legend(loc='upper right')

# Add phase labels
mid1 = CONFIG['seq_length'] // 2
mid2 = CONFIG['seq_length'] + 1 + CONFIG['seq_length'] // 2
ax1.text(mid1, -1, 'Input Phase', ha='center', fontsize=10, fontweight='bold')
ax1.text(mid2, -1, 'Output Phase', ha='center', fontsize=10, fontweight='bold')

plt.colorbar(im1, ax=ax1)

# Target
im2 = ax2.imshow(y.T.numpy(), cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax2.axvline(x=CONFIG['seq_length'], color='red', linestyle='--', linewidth=2)
ax2.set_ylabel('Output Dimensions', fontsize=11)
ax2.set_xlabel('Time Step', fontsize=11)
ax2.set_title('Target Output', fontsize=13, fontweight='bold')

# Add phase labels
ax2.text(mid1, -1, 'Zeros (ignore)', ha='center', fontsize=10, fontweight='bold')
ax2.text(mid2, -1, 'Copy Sequence', ha='center', fontsize=10, fontweight='bold')

plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("The NTM must:")
print("1. Store input vectors in memory")
print("2. Recognize the delimiter (last input dimension)")
print("3. Reproduce the sequence from memory")

## 10. Training the NTM

Now we'll train the NTM to learn the copy task using PyTorch Lightning.

### Set up trainer and train

We use gradient clipping since NTMs can have unstable gradients.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping

# Create fresh model
model = NeuralTuringMachine(
    input_size=CONFIG['vector_size'] + 1,
    output_size=CONFIG['vector_size']
)

logger = CSVLogger('logs', name='ntm_copy')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['early_stop_patience'],
    mode='min',
    verbose=False
)

trainer = L.Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='auto',
    devices=1,
    logger=logger,
    callbacks=[early_stop],
    gradient_clip_val=CONFIG['gradient_clip_val'],
    enable_progress_bar=True
)

print("Training NTM on copy task...")
trainer.fit(model, train_loader, val_loader)

### Plot training curves

Visualize learning progress.

In [ ]:
import pandas as pd

# Read metrics
metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')

# Aggregate by epoch
train_metrics = metrics[['epoch', 'train_loss', 'train_acc']].dropna()
val_metrics = metrics[['epoch', 'val_loss', 'val_acc']].dropna()
train_metrics = train_metrics.groupby('epoch').mean().reset_index()
val_metrics = val_metrics.groupby('epoch').mean().reset_index()

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(train_metrics['epoch'], train_metrics['train_loss'],
         label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax1.plot(val_metrics['epoch'], val_metrics['val_loss'],
         label='Validation', marker='s', linewidth=2, color='#FF6B6B')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    stop_epoch = trainer.early_stopping_callback.stopped_epoch
    ax1.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.5, label='Early Stop')

# Accuracy
ax2.plot(train_metrics['epoch'], train_metrics['train_acc'] * 100,
         label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax2.plot(val_metrics['epoch'], val_metrics['val_acc'] * 100,
         label='Validation', marker='s', linewidth=2, color='#FF6B6B')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    ax2.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.5, label='Early Stop')

plt.tight_layout()
plt.show()

# Report statistics
epochs_trained = len(train_metrics)
print(f"\nTraining Statistics:")
print(f"  Epochs trained: {epochs_trained} / {CONFIG['max_epochs']}")
if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    print(f"  Early stopping triggered at epoch {stop_epoch}")
print(f"\nFinal Results:")
print(f"  Train Accuracy: {train_metrics['train_acc'].iloc[-1]*100:.2f}%")
print(f"  Val Accuracy: {val_metrics['val_acc'].iloc[-1]*100:.2f}%")

## 11. Evaluating the NTM

Let's test the trained NTM on new sequences and visualize its predictions.

### Test on a new sequence

Generate predictions and compare to ground truth.

In [ ]:
model.eval()

# Get a test sample  
x, y = val_dataset[0]

# Use model's device
model_device = next(model.parameters()).device
x_batch = x.unsqueeze(0).to(model_device)

with torch.no_grad():
    y_pred = torch.sigmoid(model(x_batch))

y_pred = y_pred[0].cpu()
y_pred_binary = (y_pred > 0.5).float()

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# Input
im0 = axes[0].imshow(x.T.numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[0].axvline(x=CONFIG['seq_length'] - 0.5, color='red', linestyle='--', linewidth=2)
axes[0].set_ylabel('Input Dims', fontsize=11)
axes[0].set_title('Input Sequence', fontsize=13, fontweight='bold')
axes[0].set_xticks([])
plt.colorbar(im0, ax=axes[0])

# Target
im1 = axes[1].imshow(y.T.numpy(), cmap='Greens', aspect='auto', vmin=0, vmax=1)
axes[1].axvline(x=CONFIG['seq_length'], color='red', linestyle='--', linewidth=2)
axes[1].set_ylabel('Output Dims', fontsize=11)
axes[1].set_title('Target Output', fontsize=13, fontweight='bold')
axes[1].set_xticks([])
plt.colorbar(im1, ax=axes[1])

# Prediction
im2 = axes[2].imshow(y_pred_binary.T.numpy(), cmap='Greens', aspect='auto', vmin=0, vmax=1)
axes[2].axvline(x=CONFIG['seq_length'], color='red', linestyle='--', linewidth=2)
axes[2].set_ylabel('Output Dims', fontsize=11)
axes[2].set_xlabel('Time Step', fontsize=11)
axes[2].set_title('NTM Prediction', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

# Calculate accuracy
output_phase = slice(CONFIG['seq_length'] + 1, None)
acc = (y_pred_binary[output_phase] == y[output_phase]).float().mean()
print(f"Accuracy on output phase: {acc*100:.2f}%")
print(f"Perfect sequence copy: {'Yes' if acc == 1.0 else 'No'}")

## 12. Visualizing Memory Access Patterns

The most interesting part: let's see **how** the NTM uses its memory. We'll modify the forward pass to track addressing weights.

### Extract memory access patterns

Run the model and record read/write weights at each timestep.

In [ ]:
def forward_with_attention(model, x):
    """Forward pass that tracks attention weights."""
    model.eval()
    batch_size, seq_len, _ = x.shape
    
    # Initialize
    memory, read_weights, write_weights = model.init_state(batch_size)
    controller_state = None
    
    # Track weights over time
    read_weights_history = []
    write_weights_history = []
    outputs = []
    
    with torch.no_grad():
        for t in range(seq_len):
            # Read
            reads = [read_memory(memory, w) for w in read_weights]
            
            # Controller
            controller_input = torch.cat([x[:, t], *reads], dim=1).unsqueeze(1)
            controller_output, controller_state = model.controller(controller_input, controller_state)
            controller_output = controller_output.squeeze(1)
            
            # Update heads
            read_weights = [
                head(controller_output, memory, prev_w)
                for head, prev_w in zip(model.read_heads, read_weights)
            ]
            write_weights = model.write_head(controller_output, memory, write_weights)
            
            # Store weights
            read_weights_history.append(read_weights[0].cpu())  # First read head
            write_weights_history.append(write_weights.cpu())
            
            # Write
            erase = torch.sigmoid(model.fc_erase(controller_output))
            add = torch.tanh(model.fc_add(controller_output))
            memory = write_memory(memory, write_weights, erase, add)
            
            # Output
            reads = [read_memory(memory, w) for w in read_weights]
            output_input = torch.cat([controller_output, *reads], dim=1)
            output = model.fc_output(output_input)
            outputs.append(output)
    
    outputs = torch.stack(outputs, dim=1)
    read_weights_history = torch.stack(read_weights_history, dim=1)  # (batch, seq_len, N)
    write_weights_history = torch.stack(write_weights_history, dim=1)
    
    return outputs, read_weights_history, write_weights_history

# Extract patterns
x, y = val_dataset[0]

# Use model's device
model_device = next(model.parameters()).device
x_batch = x.unsqueeze(0).to(model_device)

outputs, read_weights, write_weights = forward_with_attention(model, x_batch)

print(f"Read weights shape: {read_weights.shape}")
print(f"Write weights shape: {write_weights.shape}")

### Visualize read and write patterns

This reveals how the NTM navigates memory over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Read weights
im1 = ax1.imshow(
    read_weights[0].T.numpy(),
    cmap='YlOrRd',
    aspect='auto',
    interpolation='nearest'
)
ax1.axvline(x=CONFIG['seq_length'] - 0.5, color='blue', linestyle='--', linewidth=2, label='Delimiter')
ax1.axvline(x=CONFIG['seq_length'], color='blue', linestyle='--', linewidth=2)
ax1.set_ylabel('Memory Location', fontsize=11)
ax1.set_title('Read Head Attention Over Time', fontsize=13, fontweight='bold')
ax1.set_xticks([])
ax1.legend(loc='upper right')
plt.colorbar(im1, ax=ax1, label='Attention Weight')

# Write weights
im2 = ax2.imshow(
    write_weights[0].T.numpy(),
    cmap='YlGnBu',
    aspect='auto',
    interpolation='nearest'
)
ax2.axvline(x=CONFIG['seq_length'] - 0.5, color='blue', linestyle='--', linewidth=2, label='Delimiter')
ax2.axvline(x=CONFIG['seq_length'], color='blue', linestyle='--', linewidth=2)
ax2.set_ylabel('Memory Location', fontsize=11)
ax2.set_xlabel('Time Step', fontsize=11)
ax2.set_title('Write Head Attention Over Time', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')
plt.colorbar(im2, ax=ax2, label='Attention Weight')

plt.tight_layout()
plt.show()

print("Observations:")
print("- Write head should be active during input phase (left of delimiter)")
print("- Read head should be active during output phase (right of delimiter)")
print("- Sequential access pattern indicates the NTM learned to use memory locations in order")

### Visualize attention focus over time

Show where the heads are looking at each timestep.

In [ ]:
# Select a few timesteps to visualize
timesteps = [
    0,  # Beginning
    CONFIG['seq_length'] // 2,  # Middle of input
    CONFIG['seq_length'],  # After delimiter
    CONFIG['seq_length'] + 1 + CONFIG['seq_length'] // 2,  # Middle of output
]

fig, axes = plt.subplots(2, len(timesteps), figsize=(16, 6))

for i, t in enumerate(timesteps):
    # Read weights
    axes[0, i].bar(
        range(CONFIG['memory_size']),
        read_weights[0, t].numpy(),
        color='#FF6B6B'
    )
    axes[0, i].set_ylim(0, 0.5)
    axes[0, i].set_xlabel('Memory Location', fontsize=9)
    axes[0, i].set_ylabel('Weight', fontsize=9)
    axes[0, i].set_title(f'Read t={t}', fontsize=11, fontweight='bold')
    axes[0, i].grid(True, alpha=0.3, axis='y')
    
    # Write weights
    axes[1, i].bar(
        range(CONFIG['memory_size']),
        write_weights[0, t].numpy(),
        color='#4ECDC4'
    )
    axes[1, i].set_ylim(0, 0.5)
    axes[1, i].set_xlabel('Memory Location', fontsize=9)
    axes[1, i].set_ylabel('Weight', fontsize=9)
    axes[1, i].set_title(f'Write t={t}', fontsize=11, fontweight='bold')
    axes[1, i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("The heads show focused attention at specific memory locations.")
print("This demonstrates learned addressability - not random access!")

## 13. Key Takeaways

**What we learned:**

1. **External memory extends neural networks** beyond fixed hidden states
   - Memory provides explicit storage separate from parameters
   - Enables learning algorithms that require memory manipulation

2. **Content-based addressing** works like attention
   - Find memory locations by similarity (cosine distance)
   - Key strength β controls focus sharpness

3. **Location-based addressing** enables sequential access
   - Interpolation blends content and location
   - Convolutional shift moves attention left/right
   - Sharpening increases focus

4. **Read and write operations** are differentiable
   - Reading: weighted sum over memory
   - Writing: erase then add with soft attention
   - All operations trainable via backpropagation

5. **NTMs can learn algorithms**
   - Copy task requires sequential write then read
   - Memory access patterns show learned structure
   - More complex tasks: sorting, associative recall, priority queues

**Limitations:**

- Training can be unstable (needs gradient clipping)
- Slower than standard RNNs due to memory operations
- Memory size is fixed (can't grow dynamically)

**Extensions:**

- **Differentiable Neural Computers (DNC)**: More sophisticated addressing
- **Sparse Access Memory (SAM)**: Efficient for large memories
- **Memory Networks**: Alternative memory-augmented architecture

## Further Exploration

Try these experiments:

1. **Longer sequences**: Increase `seq_length` - does the NTM generalize?
2. **Multiple heads**: Set `num_heads=2` - do different heads specialize?
3. **Larger memory**: Increase `memory_size` - does it use all locations?
4. **Repeat copy**: Input the same sequence multiple times - can it count?
5. **Associative recall**: Store key-value pairs, retrieve by key

The NTM is a foundational architecture that showed neural networks can learn to use memory explicitly, paving the way for modern memory-augmented models.